```markdown
>[!INFO]
> Mimo iż praca jest pisana w języku Polskim, z racji na konwencje przyjęte w branży, całość kodu zapisana jest w języku angielskim
```

# Benchmark techniczny modeli detekcji obiektów dla zadania pomiaru ruchu drogowego
Celem notebooka jest porównanie wybranych modeli detekcji obiektów pod kątem ich przydatności jako pierwszego etapu systemu pomiaru ruchu drogowego z nagrań wideo. Na tym etapie oceniane są: poprawność techniczna uruchomienia modelu, czas inferencji, liczba wykrytych obiektów należących do klas ruchu drogowego oraz możliwość ujednolicenia wyników do wspólnego formatu.

Notebook nie ocenia jeszcze pełnej dokładności pomiaru ruchu drogowego, ponieważ nie obejmuje trackingu, zliczania przekroczeń linii pomiarowej ani porównania z anotacjami referencyjnymi.

---

### Zestawienie modeli detekcji wziętych pod uwagę

| Nazwa modelu | Architektura modelu | Właściciel praw | Licencja | Dataset treningowy | AP (wg. oficjalnego źródła) |
| -- | -- | -- | -- | -- | -- |
| fasterrcnn_resnet50_fpn | CNN (ResNet-50 + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 37.0 |
| fasterrcnn_mobilenet_v3_large_fpn | CNN (MobileNetV3-Large + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 32.8 |
| retinanet_resnet50_fpn | CNN (ResNet-50 + FPN, RetinaNet) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 36.4 |
| retinanet_resnet50_fpn_v2 | CNN (ResNet-50 + FPN, RetinaNet v2) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 41.5 |
| fcos_resnet50_fpn | CNN (ResNet-50 + FPN, anchor-free FCOS) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 39.2 |
| ssd300_vgg16 | CNN (VGG16, SSD300) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 25.1 |
| ssdlite320_mobilenet_v3_large | CNN (MobileNetV3-Large, SSDLite320) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 21.3 |
| facebook/detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Facebook / Meta AI| Apache-2.0 | COCO 2017 | 42.0 |
| microsoft/conditional-detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Microsoft | Apache-2.0 | COCO 2017 | — |
| SenseTime/deformable-detr | Hybrydowa (CNN backbone + Transformer) | SenseTime | Apache-2.0 | COCO 2017 | — |
| PekingU/rtdetr_r18vd | Hybrydowa (CNN backbone + Transformer) | Peking University (PekingU) | Apache-2.0 | COCO train2017 | 46.5 |

### Przygotowanie środowiska

In [ ]:
# Import all neccessary libraries
import os
import sys
import time
import psutil
import torch
import numpy as np
import PIL.Image as Image
from pathlib import Path
from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

# silence huggingface logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
disable_progress_bars()

# silence transformers logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()


# import pyvision models
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, fasterrcnn_mobilenet_v3_large_fpn,
    retinanet_resnet50_fpn, retinanet_resnet50_fpn_v2,
    fcos_resnet50_fpn, ssd300_vgg16, ssdlite320_mobilenet_v3_large
)
from torchvision.models import list_models, get_model, get_model_weights
# import transformers utils
from transformers import AutoModelForObjectDetection, AutoImageProcessor

# mount google drive test data
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

MAPILLARY_TEST_IMAGE = Path('/content/drive/MyDrive/colab/mapilary_test_data/')
REAL_LIFE_TEST_VIDEO = Path('/content/drive/MyDrive/colab/test_data/irl_video/')

INSTANCE_PATH = Path(MAPILLARY_TEST_IMAGE/"instances")
IMAGE_PATH = Path(MAPILLARY_TEST_IMAGE/"images")
CONFIG_PATH = MAPILLARY_TEST_IMAGE / "config.json"

In [ ]:
# define constant model list
MODELS = [
    "fasterrcnn_resnet50_fpn",
    "fasterrcnn_mobilenet_v3_large_fpn",
    "retinanet_resnet50_fpn",
    "retinanet_resnet50_fpn_v2",
    "fcos_resnet50_fpn",
    "ssd300_vgg16",
    "ssdlite320_mobilenet_v3_large",
    "facebook/detr-resnet-50",
    "microsoft/conditional-detr-resnet-50",
    "SenseTime/deformable-detr",
    "PekingU/rtdetr_r18vd"
]

TV_MODELS = set(list_models())
# define test environment

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG: Using device: {device}")

In [ ]:
# loading model function
def load_model(model_name, device):
    if model_name in TV_MODELS:
        # print(f"DEBUG: Loading torchvision model: {model_name}")
        weights = get_model_weights(model_name).DEFAULT
        model = get_model(model_name, weights=weights).to(device).eval()
        preprocessing = weights.transforms()

        meta = {
            "model_name": model_name,
            "type": "torchvision",
            "weights": str(weights),
            "categories": weights.meta.get("categories"),
        }
        return model, preprocessing, meta
    
    # print(f"DEBUG: Loading transformers model: {model_name}")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForObjectDetection.from_pretrained(model_name).to(device).eval()
    meta = {
        "model_name": model_name,
        "type": "transformers",
        "weights": "from pretrained",
        "categories": model.config.id2label,
    }
    return model, processor, meta

# look for meta parameters in the model metadata
# this is to check if the metadata contains the necessary information for benchmarking

def check_meta_parameters(model, model_name):
    meta_params = [
        name for name, param in model.named_parameters()
        if getattr(param, "is_meta", False)
    ]

    meta_buffers = [
        name for name, buffer in model.named_buffers()
        if getattr(buffer, "is_meta", False)
    ]

    if meta_params or meta_buffers:
        print(f"[WARNING] {model_name}: model still has meta tensors")
        print(f"Meta parameters: {len(meta_params)}")
        print(f"Meta buffers: {len(meta_buffers)}")
        print("First examples:", meta_params[:10])
        return False

    print(f"[OK] {model_name}: no meta tensors detected")
    return True 

In [ ]:
# loding model and logging metadata

loaded_models = {}

for n, model_name in enumerate(MODELS, start=1):
    try:
        model, processor, metadata = load_model(model_name, device)
        print(f"[{n}/{len(MODELS)}] Loaded {model_name} \nwith metadata: {metadata}")
        check_meta_parameters(model, model_name)
        loaded_models[model_name] = (model, processor, metadata)
    except Exception as e:
        print(f"ERROR while loading {model_name}: {e}")

### Logika detekcji

In [43]:
# define objects of interest for benchmarking and select their corresponding class ids from the model metadata
OBJECTS = [
    "person",
    "pedestrian",
    "bicycle",
    "bike",
    "car",
    "bus",
    "truck",
    "motorcycle",
    "motorbike",
]
detection_library = {
    "person" : "pedestrian",
    "pedestrian" : "pedestrian",
    "bicycle" : "bicycle",
    "bike" : "bicycle",
    "car" : "vehicle",
    "bus" : "vehicle",
    "truck" : "vehicle",
    "motorcycle" : "motorcycle",
    "motorbike" :  "motorcycle",
}

def get_object_id(metadata, objects=OBJECTS):
    model_name = metadata["model_name"]
    categories = metadata["categories"]
    selected_ids = {}
    if isinstance(categories, list):
        iterable = enumerate(categories)
    elif isinstance(categories, dict):
        iterable = categories.items()
    else:
        print(f"[ERROR] {model_name}: unknown categories format: {type(categories)}")
    
    for class_id, class_name in iterable:
        class_id = int(class_id)
        class_name = class_name.lower().strip()
        if class_name in objects:
            selected_ids[class_id] = class_name
    return selected_ids

In [45]:
# define method for objrect detection on the still image

def detect_from_image(model_name, model_bundle, device, image_path, objects=OBJECTS):
    curr_model, processor, metadata = model_bundle
    # print(f"DEBUG: Model name {model_name}")

    object_ids = get_object_id(metadata)
    
    image = Image.open(image_path).convert("RGB")
    processed_image = processor(image).to(device)

    if model_name in TV_MODELS:
        with torch.inference_mode():
            prediction = curr_model([processed_image])[0]
            detections = []

        for box, label_id, score in zip(
            prediction["boxes"],
            prediction["labels"],
            prediction["scores"],
        ):
            label_id = int(label_id)
            score = float(score)

            if label_id not in object_ids:
                continue
            if score < 0.5:
                continue

            detections.append({
                "class_id": label_id,
                "class_name": metadata["categories"][label_id],
                "category": detection_library[metadata["categories"][label_id]],
                "score": score,
                "box": box.detach().cpu().tolist(),
            })
    else:
        detections=[]
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = curr_model(**inputs)
        target_sizes = torch.tensor([image.size[::-1]], device=device)
        results = processor.post_process_object_detection(
            outputs,
            threshold=0.5,
            target_sizes=target_sizes
        )[0]
        for box, label_id, score in zip(
            results["boxes"],
            results["labels"],
            results["scores"]
        ):
            if label_id not in object_ids:
                continue

            detections.append({
                "class_id": label_id,
                "class_name": metadata["categories"][label_id],
                "category": detection_library[metadata["categories"][label_id]],
                "score": score,
                "box": box.detach().cpu().tolist(),
            })

    return detections


## Opracowanie danych kontrolnych

In [28]:
import json

with open(CONFIG_PATH, "r") as f:
    mapillary_config = json.load(f)


MAPILLARY_TARGET_NAMES = {
    "human--person": "pedestrian",
    #"human--rider--bicyclist": "bicyclist",
    #"human--rider--motorcyclist": "motorcyclist",
    #"human--rider--other-rider": "rider",

    "object--vehicle--bicycle": "bicycle",
    "object--vehicle--bus": "vehicle",
    "object--vehicle--car": "vehicle",
    "object--vehicle--caravan": "vehicle",
    "object--vehicle--motorcycle": "motorcycle",
    "object--vehicle--other-vehicle": "vehicle",
    "object--vehicle--trailer": "vehicle",
    "object--vehicle--truck": "vehicle",
    "object--vehicle--wheeled-slow": "vehicle",
}


def build_mapillary_target_ids(config, target_names):
    target_ids = {}

    for class_id, label in enumerate(config["labels"]):
        label_name = label["name"]

        if label_name in target_names:
            target_ids[class_id] = {
                "readable": label["readable"],
                "name": label["name"],
                "target_class": target_names[label_name],
                "instances": label["instances"],
                "evaluate": label["evaluate"],
            }

    return target_ids


MAPILLARY_TARGET_IDS = build_mapillary_target_ids(
    mapillary_config,
    MAPILLARY_TARGET_NAMES
)

MAPILLARY_TARGET_IDS

{19: {'readable': 'Person',
  'name': 'human--person',
  'target_class': 'pedestrian',
  'instances': True,
  'evaluate': True},
 52: {'readable': 'Bicycle',
  'name': 'object--vehicle--bicycle',
  'target_class': 'bicycle',
  'instances': True,
  'evaluate': True},
 54: {'readable': 'Bus',
  'name': 'object--vehicle--bus',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 55: {'readable': 'Car',
  'name': 'object--vehicle--car',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 56: {'readable': 'Caravan',
  'name': 'object--vehicle--caravan',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 57: {'readable': 'Motorcycle',
  'name': 'object--vehicle--motorcycle',
  'target_class': 'motorcycle',
  'instances': True,
  'evaluate': True},
 59: {'readable': 'Other Vehicle',
  'name': 'object--vehicle--other-vehicle',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 60: {'readable': 'Trailer',
  'name':

In [ ]:
print(MAPILLARY_TEST_IMAGE)
print(IMAGE_PATH.exists())
print(INSTANCE_PATH.exists())
print(CONFIG_PATH.exists())

print("images:", len(list(IMAGE_PATH.glob("*"))))
print("instances:", len(list(INSTANCE_PATH.glob("*"))))

In [ ]:
# test does instance mask exists
def find_instance_mask(img_name):
    img_path = IMAGE_PATH/img_name
    instance_path = INSTANCE_PATH/f"{img_path.stem}.png"
    if instance_path.exists():
        return instance_path
    return "err"


In [29]:
#  count objects from the instace mask

def count_obj_from_instance(instance_path):
    mask = np.array(Image.open(instance_path))

    baseline_count = {
        "pedestrian": 0,
        #"bicyclist": 0,
        #"motorcyclist": 0,
        #"rider": 0,
        "bicycle": 0,
        "motorcycle": 0,
        "vehicle": 0,
    }

    for instance_value in np.unique(mask):
        if instance_value == 0:
            continue
        class_id = int(instance_value) // 256
        if class_id not in MAPILLARY_TARGET_IDS:
            continue
        target_class = MAPILLARY_TARGET_IDS[class_id]["target_class"]
        baseline_count[target_class] +=1
    baseline_count["total"] = sum(baseline_count.values())

    return baseline_count

In [93]:
# Create DataFrame of reference data for all images
import pandas as pd
headers = ["image", "pedestrian", "bicycle", "motorcycle", "vehicle", "total"]
rows = []
for image_name in os.listdir(INSTANCE_PATH):
    image_key = Path(image_name).stem
    c = count_obj_from_instance(INSTANCE_PATH/image_name)
    rows.append([image_key, c["pedestrian"], c["bicycle"], c["motorcycle"], c["vehicle"], c['total']])
rd = pd.DataFrame(rows, columns=headers)
REFERENCE_DATA = rd

In [98]:
def detection_runner():
    model_result = {}
    for img in os.listdir(IMAGE_PATH):
        image_key = Path(img).stem
        model_result[image_key] = {}
        for model_name, model_bundle in loaded_models.items():
            d = detect_from_image(
                model_name= model_name,
                model_bundle= model_bundle,
                device= device,
                image_path= IMAGE_PATH/img
            )
            model_result[image_key][model_name] = [[det["category"], det["score"]] for det in d]
    return model_result

results = detection_runner()

In [106]:
rows = []
header = ["image"] + ["reference"] + [model for model in MODELS]
for image in results:

    total = []
    for name in results[image]:
        total.append(len(results[image][name]))
    rows.append([image] + [int(REFERENCE_DATA.loc[REFERENCE_DATA["image"]==image, "total"].iloc[0])] +  [item for item in total])

for row in rows:
    print(row)

df = pd.DataFrame(rows, columns=header)
df.to_csv("test_data.csv", index=False)

['_6lYRZS5jTuojhqapjEL1Q', 11, 13, 6, 6, 6, 12, 1, 4, 0, 0, 0, 0]
['_bhU68GxCtat1Izwx5Tq0Q', 7, 3, 2, 1, 1, 2, 1, 1, 0, 0, 0, 0]
['_1Gn_xkw7sa_i9GU4mkxxQ', 10, 12, 13, 7, 8, 12, 3, 3, 0, 0, 0, 0]
['_92GTJv0x4fIix8B2hqahg', 15, 9, 5, 5, 7, 6, 3, 3, 0, 0, 0, 0]
['_JKIMMl114GOW9fUnSZVzQ', 22, 17, 8, 10, 9, 10, 4, 3, 0, 0, 0, 0]
['_JjvxmD9o0SSoYGIbidGoQ', 6, 10, 7, 3, 5, 5, 1, 0, 0, 0, 0, 0]
['_F2SFIm0YzTzrDuuDZP_BA', 4, 26, 10, 5, 7, 11, 1, 0, 0, 0, 0, 0]
['_J6Ge9plcquFpJng9IXIAQ', 19, 21, 6, 10, 10, 14, 3, 1, 0, 0, 0, 0]
['_4ycMRRZcIpTeDT8MqtIDw', 13, 7, 5, 4, 9, 4, 2, 2, 0, 0, 0, 0]
['_aTpHvBraCMhi_wxJ3kZSw', 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
['_dbyjJHdIT2zO20gmqt8gA', 15, 7, 1, 4, 2, 4, 1, 1, 0, 0, 0, 0]
['_3JWmnl2ILeemJ9qoxHJCg', 8, 11, 9, 9, 7, 8, 5, 5, 0, 0, 0, 0]
['_4SjTmQ-zn3XSv4D1-Tg4w', 35, 22, 15, 13, 10, 12, 4, 2, 0, 0, 0, 0]
['_gcgraos355QHyu7iFMK1w', 14, 11, 10, 8, 8, 10, 2, 0, 0, 0, 0, 0]
['_EGt7GTUW6OKw0zzz7c5-A', 6, 9, 5, 5, 4, 7, 0, 0, 0, 0, 0, 0]
['_K1nlMqMNwmPJb7D_ma